In [ ]:
import pandas as pd
import numpy as np
import re

# =============================================================================
# TABLA DE FUNCIONES
# =============================================================================
# Función               | Qué hace                                   | Se usa para
# ----------------------|--------------------------------------------|------------------------------
# limpiar_texto         | Quita espacios extra y unifica texto        | nombres, ciudad, categoría
# normalizar_nombre     | Pone nombres en formato bonito              | nombres
# validar_email         | Revisa si el correo tiene formato válido    | emails
# normalizar_telefono   | Deja teléfonos en un formato estándar       | teléfonos
# normalizar_texto_simple | Limpia texto y lo pasa a minúsculas       | ciudad, categoría
# aplicar_mapping       | Corrige valores escritos diferente         | ciudad, categoría
# validar_edad          | Verifica si la edad está en rango correcto  | edades
# validar_monto         | Verifica si el monto es razonable           | compras
# zscore_outliers       | Detecta y quita valores muy raros           | montos

np.random.seed(42)

# =========================
# 1. DATASET DESORDENADO
# =========================
datos = {
    "id_cliente": ["C001", "C002", "C003", "C003", "C005", "C006", "C007", "C008", "C009", "C010"],
    "nombre": [" Juan Perez ", "maria garcia", "CARLOS   rodriguez", "Carlos Rodriguez", "Ana Lopez", "ana  lopez", "Luis Martinez", "luis martinez", "Pedro Gomez", "pedro gomez"],
    "email": ["juan@email.com ", "maria@email", "carlos@email.com", "carlos@email.com", "ana@email.com", "ana@email.com  ", "luis@email.com", "luis@email.com", "pedro@email", "pedro@email.com"],
    "edad": [25, 31, 150, 28, np.nan, 22, 45, -3, 29, 40],
    "ciudad": [" Santiago", "santiago", "VALPARAISO", "Valparaíso", "Concepcion", "concepcion ", "la serena", "LA SERENA", "Antofagasta", "antofagasta"],
    "telefono": ["+56 9 12345678", "56912345679", "1234567", "+569 98765432", " 569-11112222 ", np.nan, "+56 9 33334444", "569 55556666", "abc123", "+56977778888"],
    "categoria": ["Electronica", "electronica", "ELECTRONICA", "Ropa", "ropa", "ROPA", "Alimentos", "alim entos", "Hogar", "Jueggos"],
    "monto_compra": [120000, 50000, -1000, 75000, 0, 25000, 9999999, 34000, 56000, 45000]
}

df = pd.DataFrame(datos)

print("DATASET DESORDENADO")
print(df)

# =========================
# 2. FUNCIONES DE LIMPIEZA
# =========================

def limpiar_texto(texto):
    """
    Quita espacios extra y deja el texto más uniforme.
    Útil para nombres, ciudad y categoría.
    """
    if pd.isna(texto):
        return np.nan
    texto = str(texto).strip()
    texto = re.sub(r"\s+", " ", texto)
    return texto

def normalizar_nombre(nombre):
    """
    Limpia el nombre y lo deja con mayúscula inicial en cada palabra.
    Ejemplo: ' juan   perez ' -> 'Juan Perez'
    """
    if pd.isna(nombre):
        return np.nan
    nombre = limpiar_texto(nombre)
    nombre = nombre.lower()
    nombre = nombre.title()
    return nombre

def validar_email(email):
    """
    Valida que el email tenga formato correcto usando regex.
    Ejemplo válido: nombre@dominio.com
    """
    if pd.isna(email):
        return np.nan
    email = limpiar_texto(email).lower().replace(" ", "")
    patron = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"
    return email if re.match(patron, email) else np.nan

def normalizar_telefono(telefono):
    """
    Limpia el teléfono y lo deja en formato estándar.
    Elimina símbolos y revisa si tiene la cantidad correcta de números.
    """
    if pd.isna(telefono):
        return np.nan
    telefono = str(telefono)
    numeros = re.sub(r"\D", "", telefono)
    if len(numeros) == 9:
        return "+56" + numeros
    if len(numeros) == 11 and numeros.startswith("56"):
        return "+" + numeros
    return np.nan

def normalizar_texto_simple(valor):
    """
    Limpia texto general:
    - quita espacios extra
    - pasa a minúsculas
    - deja el texto más fácil de comparar
    """
    if pd.isna(valor):
        return np.nan
    valor = limpiar_texto(valor)
    valor = valor.lower()
    valor = re.sub(r"\s+", " ", valor)
    return valor

mapping_ciudades = {
    "santiago": "Santiago",
    "valparaiso": "Valparaíso",
    "concepcion": "Concepción",
    "la serena": "La Serena",
    "antofagasta": "Antofagasta"
}

mapping_categorias = {
    "electronica": "Electrónica",
    "ropa": "Ropa",
    "alimentos": "Alimentos",
    "hogar": "Hogar",
    "jueggos": "Juegos",
    "juegos": "Juegos"
}

def aplicar_mapping(valor, mapping):
    """
    Corrige valores escritos de distintas formas usando un diccionario.
    Ejemplo: 'santiago', 'SANTIAGO', ' Santiago ' -> 'Santiago'
    """
    if pd.isna(valor):
        return np.nan
    valor = normalizar_texto_simple(valor)
    return mapping.get(valor, valor.title())

def validar_edad(edad):
    """
    Revisa si la edad está en un rango lógico.
    Si no está entre 18 y 100, la marca como nula.
    """
    if pd.isna(edad):
        return np.nan
    edad = float(edad)
    if 18 <= edad <= 100:
        return edad
    return np.nan

def validar_monto(monto):
    """
    Revisa si el monto de compra es positivo y razonable.
    Si está fuera del rango, lo marca como nulo.
    """
    if pd.isna(monto):
        return np.nan
    monto = float(monto)
    if 1 <= monto <= 1000000:
        return monto
    return np.nan

def zscore_outliers(df, columna, umbral=2):
    """
    Detecta valores raros con Z-score.
    Si un valor se aleja demasiado del promedio, se considera outlier.
    """
    media = df[columna].mean()
    std = df[columna].std()
    z = (df[columna] - media) / std
    return df[z.abs() <= umbral]

# =========================
# 3. LIMPIEZA
# =========================

df_limpio = df.copy()

df_limpio["nombre"] = df_limpio["nombre"].apply(normalizar_nombre)
df_limpio["email"] = df_limpio["email"].apply(validar_email)
df_limpio["telefono"] = df_limpio["telefono"].apply(normalizar_telefono)
df_limpio["ciudad"] = df_limpio["ciudad"].apply(lambda x: aplicar_mapping(x, mapping_ciudades))
df_limpio["categoria"] = df_limpio["categoria"].apply(lambda x: aplicar_mapping(x, mapping_categorias))
df_limpio["edad"] = df_limpio["edad"].apply(validar_edad)
df_limpio["monto_compra"] = df_limpio["monto_compra"].apply(validar_monto)

# Eliminar duplicados
df_limpio = df_limpio.drop_duplicates()

# Quitar outliers de monto_compra con Z-score
df_sin_outliers = df_limpio.dropna(subset=["monto_compra"]).copy()
df_sin_outliers = zscore_outliers(df_sin_outliers, "monto_compra", umbral=2)

print("\nDATASET LIMPIO")
print(df_limpio)

print("\nDATASET SIN OUTLIERS")
print(df_sin_outliers)

# =========================
# 4. RESUMEN
# =========================
print("\nRESUMEN")
print("Filas originales:", len(df))
print("Filas después de limpieza:", len(df_limpio))
print("Filas sin outliers:", len(df_sin_outliers))
print("Emails inválidos:", df_limpio["email"].isna().sum())
print("Edades inválidas:", df_limpio["edad"].isna().sum())
print("Teléfonos inválidos:", df_limpio["telefono"].isna().sum())